## РК2 Удалова Виктория Александровна ИУ5-65Б 

### 16 вариант - Метод опорных векторов, Градиентный бустинг

Датасет: https://www.kaggle.com/datasets/san-francisco/sf-restaurant-scores-lives-standard

### Тема: Методы построения моделей машинного обучения.

Задание. 
Для заданного набора данных (по Вашему варианту) постройте модели классификации или регрессии
(в зависимости от конкретной задачи, рассматриваемой в наборе данных).
Для построения моделей используйте методы 1 и 2 (по варианту для Вашей группы).
Оцените качество моделей на основе подходящих метрик качества (не менее двух метрик).
Какие метрики качества Вы использовали и почему? Какие выводы Вы можете сделать о качестве построенных моделей?
Для построения моделей необходимо выполнить требуемую предобработку данных: заполнение пропусков, кодирование категориальных признаков, и т.д.

### Описание датасета restaurant.csv :
- **business_id**: Уникальный идентификатор заведения.
- **business_name**: Название заведения.
- **business_address**: Адрес заведения.
- **business_city**: Город (San Francisco).
- **business_state**: Штат (CA).
- **business_postal_code**: Почтовый индекс.
- **business_latitude** и **business_longitude**: Координаты.
- **business_location**: JSON с координатами.
- **business_phone_number**: Телефон.
- **inspection_id**: ID проверки.
- **inspection_date**: Дата проверки.
- **inspection_score**: Оценка (выше = лучше).
- **inspection_type**: Тип проверки.
- **violation_id**: ID нарушения.
- **violation_description**: Описание нарушения.
- **risk_category**: Уровень риска ("High", "Moderate", "Low").
- **Neighborhoods (old)**: Районы (старые названия).
- **Police Districts** и др.: Административные данные.

В данных есть столбец inspection_score, который содержит числовые значения (оценки инспекции). 
Это делает задачу подходящей для регрессии, так как мы можем предсказывать числовое значение оценки.

### **Построение моделей регрессии для набора данных о ресторанах Сан-Франциско**
**Импорт библиотек**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Загрузка и предварительный анализ данных

In [2]:
data = pd.read_csv('restaurant-scores-lives-standard.csv')

### Проанализируем данные и пропуски

In [3]:
print("\nПропущенные значения:")
print(data.isnull().sum())


Пропущенные значения:
business_id                      0
business_name                    0
business_address                 0
business_city                    0
business_state                   0
business_postal_code          1018
business_latitude            19556
business_longitude           19556
business_location            19556
business_phone_number        36938
inspection_id                    0
inspection_date                  0
inspection_score             13610
inspection_type                  0
violation_id                 12870
violation_description        12870
risk_category                12870
Neighborhoods (old)          19594
Police Districts             19594
Supervisor Districts         19594
Fire Prevention Districts    19646
Zip Codes                    19576
Analysis Neighborhoods       19594
dtype: int64


In [4]:
# Удаление дубликатов (если есть)
data = data.drop_duplicates()

# Выбор признаков и целевой переменной
features = data.drop(columns=['inspection_score'])
target = data['inspection_score']

# Проверка пропусков в целевой переменной
print("Пропуски в целевой переменной:", target.isnull().sum())

# Удаление строк, где целевая переменная отсутствует
data = data.dropna(subset=['inspection_score'])
target = data['inspection_score']
features = data.drop(columns=['inspection_score'])

Пропуски в целевой переменной: 10640


## Разделение признаков на числовые и категориальные

In [5]:
# Числовые признаки (заполним медианой)
numeric_features = features.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Категориальные признаки (закодируем One-Hot)
categorical_features = features.select_dtypes(include=['object']).columns.tolist()

# Создание пайплайна для предобработки
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Заполнение пропусков медианой
    ('scaler', StandardScaler()) # Масштабирование данных
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Объединение преобразований через ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## Разделение данных на обучающую и тестовую выборки

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

### Обучение моделей
**Метод опорных векторов (SVR)**

In [7]:
svr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf', C=1.0, epsilon=0.1))
])

svr_pipeline.fit(X_train, y_train)
y_pred_svr = svr_pipeline.predict(X_test)

**Градиентный бустинг (XGBoost)**

In [8]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)

## Оценка качества моделей
**Метрики:**
Среднеквадратичная ошибка (MSE) – показывает среднюю величину ошибок в квадрате. Чем меньше, тем лучше.

Коэффициент детерминации (R²) – показывает, насколько хорошо модель объясняет дисперсию данных. Чем ближе к 1, тем лучше.

In [9]:
def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}:")
    print(f"  MSE = {mse:.2f}")
    print(f"  R² = {r2:.2f}")

evaluate_model(y_test, y_pred_svr, "SVR")
evaluate_model(y_test, y_pred_xgb, "XGBoost")

SVR:
  MSE = 46.38
  R² = 0.37
XGBoost:
  MSE = 37.09
  R² = 0.50


### Вывод
Качество моделей:

XGBoost показал лучшие результаты (меньше MSE, выше R²), чем SVR.

SVR хуже справился с задачей, возможно, из-за недостаточной настройки гиперпараметров или сложности данных.
Лучшая модель: XGBoost (MSE = 37.09, R² = 0.50).